# Coding attention mechanisms


### A simple self-attention mechanism without trainable weights

In [1]:
import torch
## this just represent a list of token embeddins.
inputs = torch.tensor([
    [0.43, 0.15, 0.89], # Your     (x^1)
    [0.55, 0.87, 0.66], # journey  (x^2)
    [0.57, 0.85, 0.64], # starts   (x^3)
    [0.22, 0.58, 0.33], # with     (x^4)
    [0.77, 0.25, 0.10], # one      (x^5)
    [0.05, 0.80, 0.55] # step     (x^6)
])

In [20]:
query = inputs[1] # this is the token embeddings for the word --journey--
attn_scores_2 = torch.empty(inputs.shape[0]) # must be equal to 6
print("attention scores 2", attn_scores_2)
for i, x_i in enumerate(inputs):    
    weight = torch.dot(query, x_i)
    attn_scores_2[i] = weight

print(attn_scores_2)

attention scores 2 tensor([0.1455, 0.2278, 0.2249, 0.1285, 0.1077, 0.1656])
tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


In [13]:
attn_weights_2_temp_normalization = attn_scores_2 / attn_scores_2.sum()
print("Attentions weights: ", attn_weights_2_temp_normalization)
print("Sum", attn_weights_2_temp_normalization.sum())

Attentions weights:  tensor([0.1455, 0.2278, 0.2249, 0.1285, 0.1077, 0.1656])
Sum tensor(1.0000)


In [14]:
def softmax_naive(x):
    return torch.exp(x) / torch.exp(x).sum(dim=0)

attn_wieghts_2_normalization_naive = softmax_naive(attn_scores_2)
print("Attention weights: ", attn_wieghts_2_normalization_naive)
print("Sum", attn_wieghts_2_normalization_naive.sum())

Attention weights:  tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum tensor(1.)


In [21]:
attn_weights_2 = torch.softmax(attn_scores_2, dim=0)
print("Attention weights:", attn_weights_2)
print("Sum:", attn_weights_2.sum())

Attention weights: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum: tensor(1.)


1. first we calculate the weight of each token related to each token in the sequence.
2. then we normalize those weight using the softmax function so that the sum up 1.
3. we calculate the context vector context_vector(n) in this case context_vector(1) by
    3.1 multipliying the weight_12 related to the current token in the sequence to its corresponding token embedding
    3.2 the result of multiplying the embedding token by it is weight related to one token in the sequence it is added to the next 
    token embedding * wieght and then we have the sum of all the weighted embeddins and that is the context vector for the token (i) 


In [25]:
query = inputs[1]
context_vector_2 = torch.zeros(query.shape)
for i, x_i in enumerate(inputs):
    context_vector_2 += attn_weights_2[i] * x_i

print("Context vector for the word: journey", context_vector_2)

Context vector for the word: journey tensor([0.4419, 0.6515, 0.5683])


In [37]:
# the result of the attention scores it is a matrix with a shape of 6*6 elements
attn_scores = torch.zeros(inputs.shape[0], inputs.shape[0])
for i, x_i in enumerate(inputs):
    for j, x_j in enumerate(inputs):
        attn_scores[i, j] = torch.dot(x_i, x_j)
print(attn_scores)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


In [35]:
inputs_transpose = inputs.T
print(inputs)
print(inputs_transpose)

tensor([[0.4300, 0.1500, 0.8900],
        [0.5500, 0.8700, 0.6600],
        [0.5700, 0.8500, 0.6400],
        [0.2200, 0.5800, 0.3300],
        [0.7700, 0.2500, 0.1000],
        [0.0500, 0.8000, 0.5500]])
tensor([[0.4300, 0.5500, 0.5700, 0.2200, 0.7700, 0.0500],
        [0.1500, 0.8700, 0.8500, 0.5800, 0.2500, 0.8000],
        [0.8900, 0.6600, 0.6400, 0.3300, 0.1000, 0.5500]])


In [57]:
# 1. compute the attention scores as dot products between the inputs.
# 2. the attentions weights are the normalized version of the attention scores
# 3. the context vectors are computed as weighted sum over the inputs. weight_1 * embedding_1 + weight_2 * embedding_2 ... weight_n + embedding_n
attn_scores = inputs @ inputs_transpose
attn_scores = torch.softmax(attn_scores, dim=-1)
print(attn_scores[1])
print(sum(attn_scores[0]))
print(attn_scores.sum(dim=-1))

tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
tensor(1.0000)
tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000])


In [58]:
# in thee third and final step, we use these attentation wieghts to compute all contexxt vectors via matrix multiplication.
all_context_vecs = attn_scores @ inputs

In [59]:
print(all_context_vecs)

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])


In [60]:
print("Context vector for the word: journey", context_vector_2)

Context vector for the word: journey tensor([0.4419, 0.6515, 0.5683])


# Implementing self-attention with trainable weights
our next step will be to implement the self-attention mechanism used in the original transformer architecture, the GPT
models, and most other popular LLMs. this self-attention mechanism is also called
**_scaled dot-product attention_** 

we will now extend the self-attention mechanism with trainable weights.

In [62]:
# we introduce 3 weights
# wq (weight for query)
# wk (weight for key)
# wv (weight for value)

x_2 = inputs[1] # represents: journey
d_in = inputs.shape[1]
d_out = 2

In [63]:
torch.manual_seed(123)
W_query = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_key = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_value = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)

In [64]:
query_2 = x_2 @ W_query
key_2 = x_2 @ W_key
value_2 = x_2 @ W_value
print(query_2)


tensor([0.4306, 1.4551])


In [67]:
keys = inputs @ W_key
values = inputs @ W_value
print("keys.shape", keys.shape)
print("values.shape", values.shape)

keys.shape torch.Size([6, 2])
values.shape torch.Size([6, 2])


In [69]:
keys_2 = keys[1]
attn_score_22 = query_2.dot(keys_2)
print(attn_score_22)

tensor(1.8524)


In [73]:
attn_score_22 = query_2 @ keys.T
print("shape of query_2", query_2.shape)
print("shape of the keys", keys.shape)

shape of query_2 torch.Size([2])
shape of the keys torch.Size([6, 2])


In [71]:
print(attn_score_22)

tensor([1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440])


In [75]:
embedding_dimesion_of_the_keys = keys.shape[-1]
print(embedding_dimesion_of_the_keys)

2


In [82]:
attn_Weights_2 = torch.softmax(attn_score_22 / embedding_dimesion_of_the_keys**0.5, dim=-1)
print(attn_score_22)
print(attn_weights_2)

tensor([1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440])
tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])


In [86]:
attn_scores_2 = query_2 @ keys.T
print(attn_scores_2)
d_k = keys.shape[-1]
print(d_k)
attn_weights_2 = torch.softmax(attn_scores_2 / d_k**0.5, dim=-1)
print(attn_weights_2)

tensor([1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440])
2
tensor([0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820])
